# Homework 13: Productization

Trains a `LinearRegression` on a generated dataset, saves it with `joblib`, then starts the
Flask API in `app.py` as a subprocess and calls both of its routes with `requests`, including
one deliberately bad call, with the live responses left in the output below.

## 1. Train and Save the Model

In [1]:
import os
import joblib
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

X, y = make_regression(n_samples=100, n_features=2, noise=0.1, random_state=42)
model = LinearRegression().fit(X, y)

os.makedirs('model', exist_ok=True)
joblib.dump(model, 'model/model.pkl')
print('Saved model/model.pkl')

Saved model/model.pkl


In [2]:
loaded = joblib.load('model/model.pkl')
sample = X[0]
print('Sample features:', sample)
print('Prediction from reloaded model:', loaded.predict([sample])[0])
print('True value for that row:', y[0])

Sample features: [-1.1913035   0.65655361]
Prediction from reloaded model: -55.863238621216084
True value for that row: -55.82011691154663


## 2. Start the API

`app.py` loads the model once at import time, not inside a route, so a request never re-reads
the file from disk. Launched here as a subprocess so this notebook can call it like any other
client would.

In [3]:
import subprocess
import sys
import time
import requests

BASE_URL = 'http://127.0.0.1:5050'

server = subprocess.Popen(
    [sys.executable, 'app.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

for _ in range(30):
    try:
        requests.get(f'{BASE_URL}/predict/0/0', timeout=1)
        print('Server is up.')
        break
    except requests.exceptions.ConnectionError:
        time.sleep(0.5)
else:
    raise RuntimeError('Server did not start in time.')

Server is up.


## 3. Call `POST /predict`

In [4]:
resp = requests.post(f'{BASE_URL}/predict', json={'features': [sample[0], sample[1]]})
print(resp.status_code, resp.json())

200 {'prediction': -55.863238621216084}


## 4. Call `GET /predict/<f1>/<f2>`

In [5]:
resp = requests.get(f'{BASE_URL}/predict/{sample[0]}/{sample[1]}')
print(resp.status_code, resp.json())

200 {'prediction': -55.863238621216084}


## 5. A Deliberately Bad Call

In [6]:
resp_bad_post = requests.post(f'{BASE_URL}/predict', json={'features': [1.0]})
print('Bad POST (wrong feature count):', resp_bad_post.status_code, resp_bad_post.json())

resp_bad_get = requests.get(f'{BASE_URL}/predict/abc/0.2')
print('Bad GET (non-numeric path param):', resp_bad_get.status_code, resp_bad_get.json())

Bad POST (wrong feature count): 400 {'error': "'features' must be a list of exactly 2 numbers"}
Bad GET (non-numeric path param): 400 {'error': 'path parameters must both be numbers'}


## 6. Shut the Server Down

In [7]:
server.terminate()
server.wait(timeout=5)
print('Server stopped.')

Server stopped.
